# Rung 10 — Self-consistency: majority vote over k sampled `number` answers

> ⚠️ **NOT RUN YET.** Cells below are the pod plan. Run them in order and **stop where marked.**

## Pre-registered rule — WRITTEN BEFORE ANY RESULT

Judged on `number` only, disaggregated ID/OOD, read as **margin** over the template-aware floor.

| Outcome | Verdict |
|---|---|
| T0 `mode_share ≈ 1.0` (no diversity) | **DEAD** — faithful negative, run nothing further |
| T0 diverse but the mode sits on the same biased value | **DEAD** — voting reinforces the error |
| `number` rises **≥ +0.02**, CI excludes 0, in ID **and** OOD | **PASS** → candidate |
| rises in one distribution only | **PARTIAL** — not promoted |
| flat or down | **FAITHFUL NEGATIVE** |

🔴 **The threshold does not move afterwards.** `bucket_mean` does NOT decide this rung — it
averages four buckets while this changes one format inside one of them.

🔴 **Why T0 comes first.** Voting moves toward the mode, and ours is *measured biased*: 05b found
81.5% of `number` errors are under-counts; 05c found true 2, 3 and 4 share the same modal
prediction (1). Aggregating a biased distribution reinforces it — the mechanism that killed
calibration. The decisive column is **`mode_closer_than_greedy`**, not entropy.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parents[1]
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "experiments" / "10-self-consistency"))

from _models.vote import diversity, t0_diversity, vote_number
from frame.config import BaselineConfig
from frame.parsing import parse_number

# Heavy run artifacts (ckpt/, merged/, inspect.csv) are gitignored, so they exist
# only in the ONE on-volume checkout that produced them — not in a fresh worktree.
# STORE is that checkout; REPO is whatever tree this notebook happens to run from.
# Keeping them apart is what lets rung 10 run on its own branch without disturbing
# the checkout another rung is training in.
STORE = Path("/workspace/repo")
RUN06 = STORE / "experiments/06-vit-lora/runs/06_vit_lora_v1"

# ckpt-1720 is the checkpoint rung 06 SELECTED (best acc_ood 0.60775 of three —
# see runs/06_vit_lora_v1/checkpoint_selection.csv); eval_best is its output.
MERGED_06 = RUN06 / "merged/checkpoint-1720"
CONTROL = RUN06 / "eval_best/inspect.csv"
PRED_06 = RUN06 / "eval_best/predictions.json"

RUN_DIR = STORE / "experiments/10-self-consistency/runs/10_self_consistency_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)

for p in (MERGED_06, CONTROL, PRED_06):
    assert p.exists(), f"missing rung 06 artifact: {p}"
print(f"REPO     {REPO}\nSTORE    {STORE}\nRUN_DIR  {RUN_DIR}")

## 1. 🔴 GATE — flag-OFF must be byte-identical

There is no local GPU, so this could not be checked before the pod. **If a single string differs,
STOP:** the flag is not defaulting OFF and the A/B is invalid.

In [ ]:
# GATE — with n_samples=1 the engine must reproduce rung 06 string for string.
# Any single mismatch means the flag is NOT defaulting OFF, the A/B compares two
# variables instead of one, and the rung stops here.
#
# Why this is a fair test of the FLAG and not of the hardware: the reference
# predictions.json was produced on THIS pod's GPU class (RTX 5090). Re-running the
# same merged checkpoint on a different card could diverge on greedy tie-breaks for
# reasons that have nothing to do with n_samples, which would make a failure
# uninterpretable. Same card => a mismatch can only come from the flag.
import json
import random
import time

from frame.data import FrameProvider, load_frame_items
from frame.engine import QwenFrameEngine

N_GATE = 50

cfg = BaselineConfig(model_path=MERGED_06)  # defaults: n_samples=1 => flag OFF
assert (cfg.n_samples, cfg.temperature, cfg.top_p) == (1, 0.0, 1.0), "defaults are not greedy"

ref = {r["qID"]: r["content"] for r in json.loads(PRED_06.read_text())}
items = load_frame_items(cfg)
assert len(items) == len(ref) == 6252, f"items={len(items)} ref={len(ref)}"

# Deterministic pick, then sorted by video so the one-video reader cache is not
# thrashed. Sorting is free: predict() depends only on (image, question), and the
# frame is a pure function of (video, round(start_time * base_fps)).
gate_items = random.Random(cfg.seed).sample(items, N_GATE)
gate_items.sort(key=lambda it: (it.dataset, it.video_id))

engine = QwenFrameEngine(cfg)
t0 = time.perf_counter()
engine.load()
t_load = time.perf_counter() - t0

provider = FrameProvider(cfg)
rows = []
for it in gate_items:
    provider.ensure_reader(it)  # untimed, as in frame.run._infer_all
    t0 = time.perf_counter()
    got = engine.predict(provider.get_frame(it), it.request.question)
    rows.append(
        {
            "qID": it.request.qID,
            "got": got,
            "want": ref[it.request.qID],
            "match": got == ref[it.request.qID],
            "latency_s": time.perf_counter() - t0,
        }
    )
provider.close()

gate = pd.DataFrame(rows)
bad = gate[~gate.match]

# Cold start is now the live latency risk: the budget is POOLED (120 s setup +
# B x 5 s), so per-question time is not the constraint but SETUP is. Row 0 here is
# an unwarmed inference, so it prices the CUDA-graph/kernel cost the real run pays
# once. Recorded, not judged — this is a 50-question gate, not a latency run.
print(f"model load        {t_load:7.1f} s")
print(f"first inference   {gate.latency_s.iloc[0]:7.2f} s  (cold)")
print(f"warm p50 / max    {gate.latency_s[1:].median():7.3f} / {gate.latency_s[1:].max():.3f} s")
print(f"GATE              {N_GATE - len(bad)}/{N_GATE} identical")

if len(bad):
    print("\nMISMATCHES:")
    print(bad[["qID", "got", "want"]].to_string(index=False))
    raise RuntimeError(f"GATE FAILED — {len(bad)}/{N_GATE} differ. Flag is not OFF; STOP.")
print("GATE PASSED — flag OFF is byte-identical to rung 06. T0 may run.")

## 2. T0 — is there anything to vote on? (~20 min)

300 `number` questions stratified by template; `temperature ∈ {0.3, 0.7, 1.0}`, k=8.
**Report and STOP.** Do not continue to T1 without an explicit go.

In [ ]:
# T0 — does sampling produce anything to vote on, and does the mode move toward truth?
# Runs ONLY after the gate passed. Reuses the engine the gate loaded (cfg is read at
# predict time, so flipping n_samples/temperature needs no reload).
import torch

T0_K = 8
T0_TEMPS = (0.3, 0.7, 1.0)
T0_PER_TEMPLATE = 38  # x8 templates ~= the 300-question budget

num = pd.read_csv(CONTROL)
num = num[num.answer_format == "number"].copy()
assert len(num) == 2094, f"expected 2094 number rows, got {len(num)}"

# Stratify by TEMPLATE. In `number` the template is the question text (8 distinct),
# and their trivial floors span 0.24..1.00 (08-data-card §3) — an unstratified draw
# would let one cheap template carry the read.
strat = num.groupby("question", group_keys=False).apply(
    lambda g: g.sample(min(len(g), T0_PER_TEMPLATE), random_state=cfg.seed)
)
print(f"T0 sample: n={len(strat)} over {strat.question.nunique()} templates, k={T0_K}")

# Frames come from FrameProvider (decord), NOT inspect.csv's `frame` column: those
# are lossy JPEGs written for eyeballing. Re-encoding is a second variable.
by_qid = {it.request.qID: it for it in items}
truth = dict(zip(strat.qID, strat.ground_truth))
greedy = dict(zip(strat.qID, strat.our_answer))

# Sorted by video so the one-video reader is opened ~40 times, not ~300 per
# temperature. Question order does not interact with the measurement: each item is an
# independent generate call, and the RNG is reseeded per temperature below.
t0_items = sorted((by_qid[q] for q in strat.qID), key=lambda it: (it.dataset, it.video_id))

records = []
provider = FrameProvider(cfg)
for temp in T0_TEMPS:
    cfg.n_samples, cfg.temperature = T0_K, temp
    torch.manual_seed(cfg.seed)  # same draw sequence if this cell is re-run
    t_start = time.time()
    for j, it in enumerate(t0_items):
        provider.ensure_reader(it)
        samples = engine.predict_samples(provider.get_frame(it), it.request.question)
        assert len(samples) == T0_K, f"expected {T0_K} samples, got {len(samples)}"
        records.append(
            {
                "qID": it.request.qID,
                "temperature": temp,
                "question": it.request.question,
                "true": truth[it.request.qID],
                "greedy": greedy[it.request.qID],
                "samples": samples,
            }
        )
        if (j + 1) % 100 == 0:
            print(
                f"  T={temp}  {j + 1}/{len(t0_items)}  "
                f"{(time.time() - t_start) / (j + 1):.2f} s/q",
                flush=True,
            )
provider.close()
cfg.n_samples, cfg.temperature = 1, 0.0  # leave the config back in its greedy default

# RAW samples first — rung 05 discarded its predicted text and 05b had to re-derive it.
records_df = pd.DataFrame(records)
records_df.to_json(RUN_DIR / "t0_samples.json", orient="records")
print(f"raw samples -> {RUN_DIR / 't0_samples.json'}")

t0 = t0_diversity(records_df)
t0.to_csv(RUN_DIR / "t0_diversity.csv", index=False)
t0

### 🛑 STOP — report T0 and wait for a go/no-go before T1

## 3. T1 — the voting run (only with a go). k and temperature are FIXED BY T0.

In [ ]:
# T1 — the voting run. CONFIG PRE-REGISTERED 2026-07-20, BEFORE ANY T1 RESULT:
#
#   arm A: k=8,  T=1.0   reference
#   arm B: k=8,  T=1.3   single variable vs A: temperature
#   arm C: k=16, T=1.0   single variable vs A: k
#
# Both contrasts are anchored on the SAME reference cell, so each moves one thing.
#
# Population = the full 2094 `number` questions. NOT stratified: T0 sub-sampled and
# its equal-per-template allocation spent 45% of the budget on templates already at
# >=0.93 where voting has no headroom. Running the whole population removes sampling
# error entirely rather than fixing the allocation.
#
# 🔴 Three arms, ONE test, NO extension after seeing the result. T0 came in at
# p=0.296 and the sample was extended *after* seeing that — doing it again here
# would be optional stopping and the p-values would not mean anything.
#
# Depends on the GATE cell (items, engine, cfg) and nothing else: T1 must be runnable
# without T0 in the same session.
import torch

T1_SMOKE = False  # smoke passed 2026-07-20 (12 q x 3 arms); this is the full run
T1_ARMS = [("A", 8, 1.0), ("B", 8, 1.3), ("C", 16, 1.0)]

num_all = pd.read_csv(CONTROL)
num_all = num_all[num_all.answer_format == "number"].copy()
assert len(num_all) == 2094, f"expected 2094 number rows, got {len(num_all)}"

by_qid = {it.request.qID: it for it in items}
truth_all = dict(zip(num_all.qID, num_all.ground_truth))
greedy_all = dict(zip(num_all.qID, num_all.our_answer))

t1_qids = list(num_all.qID)
if T1_SMOKE:
    t1_qids = t1_qids[:12]
# sorted by video: one reader open per video instead of one per question
t1_items = sorted((by_qid[q] for q in t1_qids), key=lambda it: (it.dataset, it.video_id))
print(f"T1 {'SMOKE' if T1_SMOKE else 'FULL'}: {len(t1_items)} questions x {len(T1_ARMS)} arms")

provider = FrameProvider(cfg)
for arm, k, temp in T1_ARMS:
    out = RUN_DIR / f"t1_samples_{'smoke_' if T1_SMOKE else ''}{arm}.json"
    cfg.n_samples, cfg.temperature = k, temp
    torch.manual_seed(cfg.seed)
    rows, t_start = [], time.time()
    for j, it in enumerate(t1_items):
        provider.ensure_reader(it)
        samples = engine.predict_samples(provider.get_frame(it), it.request.question)
        assert len(samples) == k, f"arm {arm}: expected {k} samples, got {len(samples)}"
        rows.append(
            {
                "qID": it.request.qID,
                "arm": arm,
                "k": k,
                "temperature": temp,
                "question": it.request.question,
                "video": it.video_id,
                "true": truth_all[it.request.qID],
                "greedy": greedy_all[it.request.qID],
                "samples": samples,
            }
        )
        if (j + 1) % 200 == 0:
            el = time.time() - t_start
            print(
                f"  {arm} k={k} T={temp}  {j + 1}/{len(t1_items)}  "
                f"{el / (j + 1):.2f} s/q  ETA {(len(t1_items) - j - 1) * el / (j + 1) / 60:.0f} min",
                flush=True,
            )
    # written per ARM, not once at the end: a crash in C must not cost A and B.
    pd.DataFrame(rows).to_json(out, orient="records")
    print(f"  arm {arm} done in {(time.time() - t_start) / 60:.1f} min -> {out.name}", flush=True)
provider.close()
cfg.n_samples, cfg.temperature = 1, 0.0  # back to the greedy default
print("T1 complete.")

## 4. T2 — scoring (zero GPU)

The control is **not** re-run — but it **is** validated first.

In [ ]:
ctrl = pd.read_csv(CONTROL)
ctrl = ctrl[ctrl.answer_format == "number"]
acc_ctrl = float(ctrl.correct.mean())
assert len(ctrl) == 2094, f"expected 2094 number rows, got {len(ctrl)}"
assert abs(acc_ctrl - 0.4250) < 0.005, f"control does not reproduce rung 06: {acc_ctrl:.4f}"
print(f"control validated: acc={acc_ctrl:.4f} over n={len(ctrl)}")

In [ ]:
# T2 — scoring. ZERO GPU: reads only the raw samples T1 wrote.
#
# Read as MARGIN over the template-aware floor, disaggregated ID/OOD, exactly as
# pre-registered. The floor CANCELS in the paired delta (same questions, same floor
# in both arms), so it does not change the verdict — it says whether an arm clears
# the trivial baseline at all, which raw accuracy hides (RULES §11).
#
# 🔴 SCOPE, per 08-data-card §"The eight `number` templates": `acc_number` over all
# 2094 "is not interpretable — do not quote it", because four of the eight templates
# admit exactly ONE answer and a fifth has no headroom (floor 0.9036). Voting cannot
# move a question whose answer never varies, so those templates only dilute. Every
# quantity is therefore reported at two scopes: `all` (the population, for
# continuity with rung 06) and `nondegenerate` (templates with >1 distinct ground
# truth — the card's "judge on the templates whose answers actually vary"). The
# scope is DERIVED from the data, never a hardcoded template list.
import datetime as _dt

from frame.metrics import _dist_from_qid, paired_delta_ci, template_floor, template_of

BAR = 0.02  # pre-registered, does not move

# Runnable from the setup cell alone — no model, no GPU.
T1_ARMS = globals().get("T1_ARMS") or [("A", 8, 1.0), ("B", 8, 1.3), ("C", 16, 1.0)]
SEED = globals()["cfg"].seed if "cfg" in globals() else BaselineConfig().seed


def score_arm(arm: str) -> pd.DataFrame:
    """One row per question with BOTH arms scored by the SAME function.

    Scoring greedy with the SDK judge and the vote with our own comparison would
    put two variables in the A/B, so both are recomputed here as exact integer
    match — and the greedy arm is then gated against the SDK's own number.
    """
    df = pd.read_json(RUN_DIR / f"t1_samples_{arm}.json")
    df["voted"] = df["samples"].map(vote_number)
    df["true_n"] = df["true"].astype(float)
    df["correct_a"] = (df["greedy"].map(parse_number) == df["true_n"]).astype(float)
    df["correct_b"] = (df["voted"] == df["true_n"]).astype(float)
    df["distribution"] = df["qID"].map(_dist_from_qid)
    df["template"] = df["question"].map(template_of)
    return df


def cell(sub: pd.DataFrame, **keys) -> dict:
    floor = template_floor(sub, answer_col="true", template_col="template")
    ci = paired_delta_ci(sub, n_boot=2000, seed=SEED)
    acc_a, acc_b = float(sub.correct_a.mean()), float(sub.correct_b.mean())
    return {
        **keys, "n": ci["n"], "n_videos": ci["n_videos"], "floor": round(floor, 4),
        "acc_greedy": round(acc_a, 4), "acc_voted": round(acc_b, 4),
        "margin_greedy": round(acc_a - floor, 4), "margin_voted": round(acc_b - floor, 4),
        "delta": round(ci["delta"], 4), "ci_low": round(ci["ci_low"], 4),
        "ci_high": round(ci["ci_high"], 4),
        "wins_voted": ci["wins_b"], "wins_greedy": ci["wins_a"],
    }


rows, tmpl_rows = [], []
for arm, k, temp in T1_ARMS:
    d = score_arm(arm)
    assert len(d) == 2094, f"arm {arm}: expected 2094 rows, got {len(d)}"

    # 🔴 GATE — our exact-match rescoring of the GREEDY arm must reproduce the SDK's
    # own number for rung 06 (0.4250). If it does not, the scoring function is wrong
    # and every delta below it is meaningless.
    acc_a_all = float(d.correct_a.mean())
    assert abs(acc_a_all - 0.4250) < 0.005, (
        f"arm {arm}: rescored greedy acc {acc_a_all:.4f} does not reproduce the SDK's "
        "0.4250 — the scoring function diverges from the judge; STOP."
    )

    nondeg = {t for t, g in d.groupby("template") if g["true"].nunique() > 1}
    for scope, dsc in (("all", d), ("nondegenerate", d[d.template.isin(nondeg)])):
        for dist in ("ID", "OOD", "overall"):
            sub = dsc if dist == "overall" else dsc[dsc.distribution == dist]
            rows.append(cell(sub, arm=arm, k=k, temperature=temp, scope=scope, distribution=dist))
    for t, g in d.groupby("template"):
        tmpl_rows.append(cell(g, arm=arm, k=k, temperature=temp,
                              n_distinct_true=int(g["true"].nunique()), template=t[:60]))

res = pd.DataFrame(rows)
res.to_csv(RUN_DIR / "RESULTS_arms.csv", index=False)
pd.DataFrame(tmpl_rows).to_csv(RUN_DIR / "RESULTS_templates.csv", index=False)
print(res.to_string(index=False))

# Pre-registered verdict, applied mechanically — the bar does not move. Judged at
# the `nondegenerate` scope, which is where the data card says the signal can live.
print()
summary = []
for arm, k, temp in T1_ARMS:
    def pick(scope, dist):
        return res[(res.arm == arm) & (res.scope == scope) & (res.distribution == dist)].iloc[0]
    a, o = pick("nondegenerate", "ID"), pick("nondegenerate", "OOD")
    ov_all, ov_nd = pick("all", "overall"), pick("nondegenerate", "overall")
    hits = [s.delta >= BAR and s.ci_low > 0 for s in (a, o)]
    verdict = "PASS" if all(hits) else "PARTIAL" if any(hits) else "FAITHFUL_NEGATIVE"
    print(
        f"arm {arm} (k={k}, T={temp}) [nondegenerate]:  "
        f"ID {a.delta:+.4f} [{a.ci_low:+.4f},{a.ci_high:+.4f}]  "
        f"OOD {o.delta:+.4f} [{o.ci_low:+.4f},{o.ci_high:+.4f}]  ->  {verdict}"
    )
    summary.append({
        "run": "10_self_consistency_v1", "arm": arm, "k": k, "temperature": temp,
        "n_all": int(ov_all.n), "n_nondegenerate": int(ov_nd.n),
        "acc_nondeg_greedy": ov_nd.acc_greedy, "acc_nondeg_voted": ov_nd.acc_voted,
        "margin_nondeg_ID": a.margin_voted, "margin_nondeg_OOD": o.margin_voted,
        "delta_ID": a.delta, "ci_low_ID": a.ci_low, "ci_high_ID": a.ci_high,
        "delta_OOD": o.delta, "ci_low_OOD": o.ci_low, "ci_high_OOD": o.ci_high,
        "acc_all_greedy": ov_all.acc_greedy, "acc_all_voted": ov_all.acc_voted,
        "verdict": verdict, "date": _dt.date.today().isoformat(),
        "notes": (
            f"majority vote over k={k} at T={temp}, full 2094 `number`; baseline = rung 06 "
            "ckpt-1720 greedy rescored identically (gated to reproduce the SDK 0.4250). "
            "Ties break LOW. Paired video-level bootstrap (frame.metrics.paired_delta_ci, "
            "n_boot=2000). VERDICT is judged at the `nondegenerate` scope (>1 distinct "
            "ground truth) per 08-data-card: acc_number over all 2094 is not interpretable. "
            "Pre-registered bar +0.02 with CI excluding 0 in ID AND OOD; 3 arms, one test, "
            "no extension. Detail: runs/10_self_consistency_v1/RESULTS_{arms,templates}.csv."
        ),
    })

# Tracked summary at the EXPERIMENT level — the file `frame.measured` indexes (it
# globs experiments/*/RESULTS*.csv). Writing only into runs/ would leave the result
# invisible to MEASURED.md, which is that index's whole job.
tracked = REPO / "experiments/10-self-consistency/RESULTS.csv"
pd.DataFrame(summary).to_csv(tracked, index=False)
print(f"\ntracked summary -> {tracked}")
print("next: rung 10 README ladder, context/decisions/<slug>.md with frontmatter, "
      "then `python -m frame.measured`.")